# Rubrik adapter'ı — ölçüm

`rubric-qlora` koşusunun ürettiği adapter'ı puanlar. Eğitimi tekrarlamaz:
adapter `kernel_sources` ile o koşunun çıktısından geliyor, yani ölçüm düşerse
beş saatlik eğitim yerinde duruyor.

Taban ve adapter **aynı oturumda**, aynı promptlarla, aynı greedy çözmeyle
ölçülüyor. Tek fark model.

## Ne kazanç sayılır

Bu base (`Qwen3-4B-Instruct-2507`) kanıt yokluğunu zaten %89 doğrulukla
söylüyor ve cevaplarının %95'i şema geçerli. Kazanç oradan gelmez:

- **`present_score_mae`** — hedef. Taban 0,77; düşmesi lazım.
- `absent_rate`, `schema_valid` — **taban**, düşerse build gider.

Önceki sürüm `absent_rate`'i hedef sayıyordu; o rakam gemma-2-2b-it'e aitti ve
base değişirken yeniden ölçülmemişti.

## İki ölçüm, ve neden ikisi de gerekli

**Held-out set** — yukarıdaki dört sayı. "Bu vakayı daha önce gördü mü"
sorusunu cevaplar.

**Contrast set** — `direction`, `stability`, `consistency`. Held-out'un
cevaplayamadığını cevaplar: *kuralı mı öğrendi, bankayı mı*. Her vaka aynı 51
fragmentten kurulduğu için model okumadan **tanıyarak** iyi puan alabilir.
Contrast çiftleri tek paragrafı değişen aynı vaka, ve tanıma bunu atlatamaz.

MAE düşüp `consistency` yükselmiyorsa kazanç muhtemelen ezberdir. Script bunu
kendi basıyor.

In [ ]:
import glob, json, os, shutil, sys
import torch

assert torch.cuda.is_available(), "GPU acik degil - Settings > Accelerator > GPU T4"
cap = torch.cuda.get_device_capability(0)
print("GPU:", torch.cuda.get_device_name(0), "sm_%d%d" % cap)
print("bellek: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

# Fail here, in five seconds, rather than after an 8 GB download. A P100 is
# sm_60: Kaggle's torch build does not support it at all, and bitsandbytes needs
# sm_75 for 4-bit NF4. The Flutter run landed on one because kernel-metadata
# omitted machine_shape, and the error arrived half an hour in wearing a
# different mask.
assert cap >= (7, 5), (
    f"sm_{cap[0]}{cap[1]} yetersiz - 4-bit NF4 icin T4 (sm_75) gerekiyor. "
    "Settings > Accelerator > GPU T4 x2")

In [ ]:
# Qwen3 icin transformers >= 4.51 gerekiyor; Kaggle imaji eskiyse sessizce
# 'unknown architecture' ile duser.
!pip -q install -U "transformers>=4.51" "peft>=0.11" "bitsandbytes>=0.43" "accelerate>=0.30" datasets 2>&1 | tail -2
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__, "| bnb", bitsandbytes.__version__)
# torchao kaldiriliyor, yukseltilmiyor. peft'in LoRA dispatcher'i sardigi her
# kuantize OLMAYAN Linear icin is_torchao_available() soruyor ve o fonksiyon
# uyumsuz surumde False donmek yerine ImportError firlatiyor. Kaggle imaji
# 0.10.0 tasiyor, peft ('peft>=0.11' artik 0.20'ye cozuluyor) >0.16.0 istiyor.
#
# Tuzak fp16 kolunda: 4-bit'te bitsandbytes kendi Linear4bit'ini once
# eslestirdigi icin dispatcher'a hic varilmiyor. colab-pilot-eval bunu bir kez
# odedi ve cozdu; buraya tasinmadigi icin rubric-curve-eval ayni duvara carpti
# — taban olcumu bittikten sonra, adapter gecisinin ilk saniyesinde.
#
# Silmek find_spec'i None yapar ve kontrol False doner, ki dogru cevap odur:
# torchao nicemlemesi kullanmiyoruz. Yukseltmek torch'u da suruklerdi.
!pip -q uninstall -y torchao 2>&1 | tail -1


In [ ]:
def find_mount(slug, marker):
    """Locate one input mount by the dataset/kernel slug in its path.

    Not by filename. A kernel attached with kernel_sources contributes the whole
    of its /kaggle/working, which for the training run includes its own copies
    of the data files and the scripts — so searching for `rubric_eval.jsonl`
    finds two mounts and picks between them by luck. The slug is the only thing
    that distinguishes them, and it appears in the path.

    Recursive on top of that, because the mount depth is not a promise: the same
    dataset has appeared directly under /kaggle/input and, on the next run, one
    level deeper under /kaggle/input/datasets.
    """
    hits = [p for p in glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
            if slug.split("/")[-1] in p]
    assert hits, (f"'{slug}' bagli degil (aranan: {marker}). "
                  f"Kaggle > Notebook > Add Input, ve surumun islenmesi bitmis olmali.")
    return os.path.dirname(sorted(hits, key=len)[0])


for root, dirs, files in os.walk("/kaggle/input"):
    print(root, "->", sorted(files)[:4], "..." if len(files) > 4 else "")
    if root.count("/") > 6:
        dirs.clear()

In [ ]:
WORK = "/kaggle/working"
DATA = find_mount("emrahik/rubric-dataset", "rubric_train.jsonl")
print("veri seti:", DATA)

os.makedirs(f"{WORK}/data", exist_ok=True)
for f in os.listdir(DATA):
    dst = f"{WORK}/data/{f}" if f.endswith(".jsonl") else f"{WORK}/{f}"
    shutil.copy(f"{DATA}/{f}", dst)
os.chdir(WORK)
print(sorted(os.listdir(WORK)))
print(sorted(os.listdir(f"{WORK}/data")))

# The adapter comes from the training kernel's output, which is the whole of its
# /kaggle/working — including its own copies of data/ and the scripts. That is
# why find_mount matches on the slug: searching for adapter_config.json alone
# would be fine here, but searching for a data file would not, and the two
# mounts are otherwise indistinguishable.
ADAPTER = find_mount("emrahik/rubric-qlora", "rubric-v1/adapter_config.json")
print("adapter:", ADAPTER)
assert os.path.isdir(ADAPTER), ADAPTER

## Taban + adapter, tek koşuda

`--limit 60` held-out satır, `--contrast-limit 20` çift. Her çift iki üretim
demek, o yüzden ayrı fiyatlanıyor.

Script önce tabanı, sonra adapter'ı ölçer ve deltayı basar. Beş kapıdan biri
düşerse söyler ve o build yayına alınmaz.

Contrast yalnızca **yatırım** setinden çekiliyor; pazarlama tarafının ezber
kontrolü bu koşuda yok, ve iki rubrik tek adapter'da olduğu için oradaki
sonuç buraya taşınamaz.

In [ ]:
import subprocess, sys

# `!` degil: cikis kodu okunmali. Egitim notebook'unda ayni kalip, CUDA OOM ile
# olen bir kosunun COMPLETE gorunmesine sebep oldu.
r = subprocess.run([sys.executable, "rubric_eval.py",
                    "--data", "data/rubric_eval.jsonl",
                    "--base-model", "Qwen/Qwen3-4B-Instruct-2507",
                    "--adapter", ADAPTER,
                    "--contrast", "data/contrast_investment.jsonl",
                    "--contrast-limit", "20",
                    "--limit", "60",
                    "--out", "out/after.json"])
assert r.returncode == 0, f"olcum coktu (exit {r.returncode}) — log yukarida"

## Sonuç

`out/after.json` iki tarafın da sayılarını taşıyor — panelde adapter kaydı
açılırken `notes` alanına konacak olan şey o.

In [ ]:
f = "out/after.json"
if os.path.exists(f):
    print(json.dumps(json.load(open(f)), indent=2, ensure_ascii=False))